<a href="https://colab.research.google.com/github/vinopeng95-dev/Clinic-Registration-System/blob/main/SDDB_Week07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📋 規格書

學號：＿411312472＿　姓名：_彭韋諾_

## 1. 系統目的（一句話）

讓使用者透過簡單的文字輸入，完成診所掛號、取消掛號及查詢剩餘名額。

## 2. 使用者會怎麼說（至少 5 句）

1. 我要掛內科。
2. 幫我掛牙科。
3. 我要取消外科掛號。
4. 內科還剩多少名額？
5. 我要掛外科。
6. 我要掛眼科。
7. 我頭痛應該掛哪一科？
8. 幫我取消牙科掛號。

## 3. 資料表（最多 2 張）

| 表名 | 欄位 | 型態 | 說明 |
|---|---|---|---|
| departments | id | INTEGER | 科別編號 |
| departments | name | TEXT | 科別名稱 |
| departments | total | INTEGER | 總名額 |
| departments | remaining | INTEGER | 剩餘名額 |
| registrations | id | INTEGER | 掛號編號 |
| registrations | department | TEXT | 掛號科別 |
| registrations | status | TEXT | 掛號狀態 |

初始資料：

| id | name | total | remaining |
|---|---|---|---|
| 1 | 內科 | 10 | 10 |
| 2 | 外科 | 10 | 10 |
| 3 | 牙科 | 10 | 10 |

掛號紀錄初始為空。

## 4. AI 的工作

- 輸入：使用者的一句話
- 輸出格式：`動作,對象`（例：`掛號,內科`）
- 動作只能是：掛號、取消、查詢
- 對象只能是：內科、外科、牙科
- 無法判斷時輸出：`未知,未知`
- AI 不可以做的事：不能提供醫療建議、不能根據症狀推薦科別、不能自行修改資料庫。

## 5. 功能與規則

| 功能 | 條件 | 系統回應 | 資料庫變化 |
|---|---|---|---|
| 掛號 | 科別正確且有名額 | 掛號成功 | 剩餘名額減 1，新增掛號紀錄 |
| 掛號 | 科別已額滿 | 掛號失敗，名額不足 | 不變 |
| 取消 | 有可取消的掛號紀錄 | 取消成功 | 剩餘名額加 1，更新掛號狀態 |
| 取消 | 沒有可取消的掛號紀錄 | 找不到掛號紀錄 | 不變 |
| 查詢 | 科別正確 | 顯示剩餘名額 | 不變 |
| 未知 | 無法判斷需求或科別不支援 | 請重新輸入正確的需求 | 不變 |

## 6. 例外情況（至少 3 個）

1. 使用者輸入不支援的科別，例如眼科，系統顯示不支援此科別。
2. 使用者要掛號的科別已經額滿，系統顯示名額不足，無法掛號。
3. 使用者取消不存在的掛號，系統顯示找不到掛號紀錄。
4. 使用者輸入不完整，例如只說「我要掛號」，系統請使用者指定科別。
5. 使用者詢問症狀，例如「我頭痛應該掛哪一科」，系統不提供醫療建議，請使用者自行指定科別。

## 7. 測試清單（至少 8 句，正常／錯誤／邊界都要有）

| # | 類型 | 使用者說的話 | 預期結果 |
|---|---|---|---|
| 1 | 正常 | 我要掛內科 | 掛號成功，內科剩餘名額減 1 |
| 2 | 正常 | 幫我掛牙科 | 掛號成功，牙科剩餘名額減 1 |
| 3 | 正常 | 內科還剩多少名額？ | 顯示內科剩餘名額 |
| 4 | 正常 | 我要取消內科掛號 | 取消成功，內科剩餘名額加 1 |
| 5 | 錯誤 | 我要掛眼科 | 顯示不支援此科別 |
| 6 | 錯誤 | 我頭痛應該掛哪一科？ | 不提供醫療建議，請使用者自行指定科別 |
| 7 | 錯誤 | 我要掛號 | 請使用者指定科別 |
| 8 | 邊界 | 內科額滿後，我要掛內科 | 顯示名額不足，無法掛號 |
| 9 | 邊界 | 我要取消沒有掛號紀錄的外科掛號 | 顯示找不到掛號紀錄 |
| 10 | 正常 | 幫我取消牙科掛號 | 若有有效掛號紀錄，則取消成功 |

In [53]:
!pip -q install groq

from google.colab import userdata
from groq import Groq

key = userdata.get("GROQ_API_KEY")

client = Groq(api_key=key)
MODEL = "openai/gpt-oss-120b"

print("✅ Groq AI 設定成功")

✅ Groq AI 設定成功


In [54]:
# 第 2 格：建立資料庫與初始資料

import sqlite3

conn = sqlite3.connect("clinic.db")
c = conn.cursor()

c.execute("DROP TABLE IF EXISTS departments")
c.execute("DROP TABLE IF EXISTS registrations")

c.execute("""
CREATE TABLE departments (
    id INTEGER PRIMARY KEY,
    name TEXT,
    total INTEGER,
    remaining INTEGER
)
""")

c.execute("""
CREATE TABLE registrations (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    department TEXT,
    status TEXT
)
""")

data = [
    (1, "內科", 10, 10),
    (2, "外科", 10, 10),
    (3, "牙科", 10, 10)
]

c.executemany(
    "INSERT INTO departments VALUES (?, ?, ?, ?)",
    data
)

conn.commit()
conn.close()

print("✅ 資料庫建立成功")

✅ 資料庫建立成功


In [55]:
import pandas as pd

def show_tables():
    conn = sqlite3.connect("clinic.db")

    try:
        a = pd.read_sql("SELECT * FROM departments", conn)
        b = pd.read_sql("SELECT * FROM registrations", conn)

        print("科別資料：")
        display(a)

        print("掛號紀錄：")
        display(b)

    finally:
        conn.close()

show_tables()

科別資料：


,id,name,total,remaining
0,1,內科,10,10
1,2,外科,10,10
2,3,牙科,10,10


掛號紀錄：


,id,department,status


In [56]:
def ai_parse(text):
    prompt = """
你是診所掛號系統的助手。

請判斷使用者的動作與科別。

動作只能是：掛號、取消、查詢
科別只能是：內科、外科、牙科

回答格式：動作,科別

例如：
我要掛內科 → 掛號,內科
我要取消牙科掛號 → 取消,牙科
外科還剩多少名額 → 查詢,外科

如果沒有指定科別、不支援該科別、
只描述症狀或無法判斷，
回答：未知,未知

不可提供醫療建議。
只回答結果，不要解釋。
"""

    try:
        r = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": text}
            ],
            temperature=0
        )

        raw = r.choices[0].message.content or ""
        parts = raw.strip().split(",")

        if len(parts) == 2:
            a = parts[0].strip()
            b = parts[1].strip()

            if a in ["掛號", "取消", "查詢"] and b in ["內科", "外科", "牙科"]:
                return a, b, raw

        return "未知", "未知", raw

    except Exception:
        return "未知", "未知", "AI 呼叫失敗"

In [57]:
tests = [
    ("我要掛內科", "掛號", "內科"),
    ("幫我掛牙科", "掛號", "牙科"),
    ("我要取消外科掛號", "取消", "外科"),
    ("內科還有多少名額", "查詢", "內科"),
    ("我要掛眼科", "未知", "未知"),
    ("我肚子痛", "未知", "未知"),
    ("我要掛號", "未知", "未知"),
    ("幫我取消牙科", "取消", "牙科")
]

correct = 0

for text, a, b in tests:
    x, y, raw = ai_parse(text)

    print("使用者：", text)
    print("AI 原始回覆：", raw)
    print("判斷結果：", x, y)

    if x == a and y == b:
        print("✅ 正確")
        correct += 1
    else:
        print("❌ 錯誤")

    print("--------------------")

print("測試結果：", correct, "/", len(tests))

使用者： 我要掛內科
AI 原始回覆： 掛號,內科
判斷結果： 掛號 內科
✅ 正確
--------------------
使用者： 幫我掛牙科
AI 原始回覆： 掛號,牙科
判斷結果： 掛號 牙科
✅ 正確
--------------------
使用者： 我要取消外科掛號
AI 原始回覆： 取消,外科
判斷結果： 取消 外科
✅ 正確
--------------------
使用者： 內科還有多少名額
AI 原始回覆： 查詢,內科
判斷結果： 查詢 內科
✅ 正確
--------------------
使用者： 我要掛眼科
AI 原始回覆： 未知,未知
判斷結果： 未知 未知
✅ 正確
--------------------
使用者： 我肚子痛
AI 原始回覆： 未知,未知
判斷結果： 未知 未知
✅ 正確
--------------------
使用者： 我要掛號
AI 原始回覆： 未知,未知
判斷結果： 未知 未知
✅ 正確
--------------------
使用者： 幫我取消牙科
AI 原始回覆： 取消,牙科
判斷結果： 取消 牙科
✅ 正確
--------------------
測試結果： 8 / 8


In [58]:
def register(dept):
    conn = sqlite3.connect("clinic.db")
    try:
        c = conn.cursor()
        c.execute("SELECT remaining FROM departments WHERE name = ?", (dept,))
        row = c.fetchone()

        if row is None:
            return "沒有這個科別"

        if row[0] <= 0:
            return dept + "已經額滿"

        c.execute(
            "UPDATE departments SET remaining = remaining - 1 WHERE name = ?",
            (dept,)
        )
        c.execute(
            "INSERT INTO registrations (department, status) VALUES (?, ?)",
            (dept, "已掛號")
        )

        conn.commit()
        return dept + "掛號成功"

    finally:
        conn.close()


def cancel(dept):
    conn = sqlite3.connect("clinic.db")
    try:
        c = conn.cursor()
        c.execute(
            "SELECT id FROM registrations WHERE department = ? AND status = ? ORDER BY id LIMIT 1",
            (dept, "已掛號")
        )
        row = c.fetchone()

        if row is None:
            return dept + "沒有可以取消的掛號"

        c.execute(
            "UPDATE registrations SET status = ? WHERE id = ?",
            ("已取消", row[0])
        )
        c.execute(
            "UPDATE departments SET remaining = remaining + 1 WHERE name = ?",
            (dept,)
        )

        conn.commit()
        return dept + "取消掛號成功"

    finally:
        conn.close()


def query(dept):
    conn = sqlite3.connect("clinic.db")
    try:
        c = conn.cursor()
        c.execute(
            "SELECT remaining FROM departments WHERE name = ?",
            (dept,)
        )
        row = c.fetchone()

        if row is None:
            return "沒有這個科別"

        return dept + "剩餘名額：" + str(row[0])

    finally:
        conn.close()

In [59]:
print(query("內科"))
print(register("內科"))
print(query("內科"))
print(cancel("內科"))
print(query("內科"))

內科剩餘名額：10
內科掛號成功
內科剩餘名額：9
內科取消掛號成功
內科剩餘名額：10


In [60]:
def main(text):
    a, b, raw = ai_parse(text)

    if a == "掛號":
        return register(b)

    elif a == "取消":
        return cancel(b)

    elif a == "查詢":
        return query(b)

    else:
        return "請直接說明要掛哪一科（內科、外科、牙科），或洽詢診所櫃台。"


print(main("我要掛內科"))
print(main("內科還剩多少名額"))
print(main("我要取消內科掛號"))
print(main("內科還剩多少名額"))
print(main("我要掛眼科"))

內科掛號成功
內科剩餘名額：9
內科取消掛號成功
內科剩餘名額：10
請直接說明要掛哪一科（內科、外科、牙科），或洽詢診所櫃台。


In [61]:
print("=== 額滿測試 ===")

for i in range(11):
    print(i + 1, register("牙科"))

print("=== 取消測試 ===")

print(cancel("外科"))
print(query("牙科"))

=== 額滿測試 ===
1 牙科掛號成功
2 牙科掛號成功
3 牙科掛號成功
4 牙科掛號成功
5 牙科掛號成功
6 牙科掛號成功
7 牙科掛號成功
8 牙科掛號成功
9 牙科掛號成功
10 牙科掛號成功
11 牙科已經額滿
=== 取消測試 ===
外科沒有可以取消的掛號
牙科剩餘名額：0


In [62]:
import gradio as gr

def chat(text):
    return main(text)

demo = gr.Interface(
    fn=chat,
    inputs=gr.Textbox(
        label="請輸入需求",
        placeholder="例如：我要掛內科"
    ),
    outputs=gr.Textbox(label="系統回覆"),
    title="診所 AI 掛號系統",
    description="提供內科、外科、牙科的掛號、取消及名額查詢服務。",
    examples=[
        ["我要掛內科"],
        ["我要取消牙科掛號"],
        ["外科還剩多少名額"],
        ["我要掛眼科"]
    ]
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2a4b95689101969087.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
